In [2]:
%pip install --upgrade langchain langchain-community langchain-classic langchain-core langchain-google-genai langchain-huggingface chromadb pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv

# Document Loaders & Splitters
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Store & Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Gemini & Chains
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

c:\Users\Nikhil\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\Nikhil\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tf_keras/protobuf/saved_metadata.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\Nikhil\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tf_keras/protobuf/versions.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(


In [4]:
# 1. Load Environment Variables 
load_dotenv()

True

In [5]:
#2. Define specific list of pdf files
pdf_files = [r"D:\Bookxpert(Technical Assignment)\python_notes.pdf",
    r"D:\Bookxpert(Technical Assignment)\SQL_notes.pdf",
    r"D:\Bookxpert(Technical Assignment)\MLBasics.pdf",
    r"D:\Bookxpert(Technical Assignment)\AI_notes.pdf"
    ]

In [6]:
# 3. Load each file individually using PyPDFLoader
all_documents = []
for file_path in pdf_files:
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    all_documents.extend(pages)
    print(f"loaded {len(pages)} pages from {file_path}")

loaded 155 pages from D:\Bookxpert(Technical Assignment)\python_notes.pdf
loaded 58 pages from D:\Bookxpert(Technical Assignment)\SQL_notes.pdf
loaded 287 pages from D:\Bookxpert(Technical Assignment)\MLBasics.pdf
loaded 154 pages from D:\Bookxpert(Technical Assignment)\AI_notes.pdf


In [7]:
# 4. Chunking Strategy
# we use RecursiveCharacterTextSplitter  with overlap to maintain context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    add_start_index = True
)

chunks = text_splitter.split_documents(all_documents)
print(f"\nTotal documents loaded: {len(pdf_files)}")
print(f"Total chunks created: {len(chunks)}")


Total documents loaded: 4
Total chunks created: 1647


In [8]:
#5.Use a local embedding model (Runs on your CPU - No API Key / No Limits)
# This model is small, fast, and excellent for document retrieval
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [9]:
persist_directory = "./chroma_db"

In [10]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [11]:
#6. verification test
query = "what is sql join?"
docs = vector_store.similarity_search(query,k=2)

In [12]:
for i,doc in enumerate(docs):
    print(f"Source: {doc.metadata['source']}")
    print(f"Snippet: {doc.page_content[:150]}...\n")

Source: D:\Bookxpert(Technical Assignment)\SQL_notes.pdf
Snippet: SQL JOIN
SQL, JOIN means "to combine two or more tables". In SQL, JOIN clause is used to
combine the records from two or more tables in a database.
Ty...

Source: D:\Bookxpert(Technical Assignment)\SQL_notes.pdf
Snippet: What is SQL Process?...



In [13]:
# 7.Initialize the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [20]:
# Define a prompt template for the RAG chain
system_prompt = (
    "You are a strict technical assistant. Use ONLY the provided context to answer. "
    "If the specific topic or entity (e.g., a specific app or brand name) mentioned in the "
    "question is not explicitly discussed in the context, you must state: "
    "'I'm sorry, but the provided documents do not contain information about [Topic].' "
    "Do not attempt to find similar-sounding words or concepts. "
    "Do not use external knowledge. \n\n"
    "Context: {context}"
)

In [21]:
prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])

In [22]:
#buld the RAG chain
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
qa_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)

In [27]:
# Interact with the RAG chain
while True:
    query = input("\nAsk a question (or type 'exit'): ")
    if query.lower() in ['exit', 'quit']: 
        break
    if not query.strip():
        continue

    response = rag_chain.invoke({"input": query})
    answer = response['answer']

    print(f"\nAnswer: {answer}")
    
    # We check for common phrases used in your system prompt
    negative_phrases = ["i'm sorry", "don't know", "not mentioned", "not contained"]

    is_not_found = any(phrase in answer.lower() for phrase in negative_phrases)

    if not is_not_found:
        # Only show citations if the answer was actually found in the documents
        sources = {os.path.basename(doc.metadata['source']) for doc in response['context']}
        print(f"Sources: {', '.join(sources)}")
    else:
        print("Sources: None (Question outside the context)")



Answer: I'm sorry, but the provided documents do not contain information about cat.
Sources: None (Question outside the context)

Answer: Tuples are immutable and usually contain a heterogeneous sequence of elements. They are accessed via unpacking, indexing, or by attribute in the case of namedtuples. It is not possible to assign to the individual items of a tuple, but they can contain mutable objects like lists.
Sources: python_notes.pdf
